# Sprint 1 — Leitura do Capítulo 1 e do Apêndice A

**Capítulo 1 — *Understanding Large Language Models*** e **Apêndice A — *Introduction to PyTorch*** ·
RASCHKA, S. *Build a Large Language Model (From Scratch)*, Manning.

Este notebook registra as respostas às questões da leitura orientada. Ele é propositalmente composto
só de células de texto: os trechos de código aparecem como exemplos dentro do texto, e a validação
prática do PyTorch fica em [`src/verificar_ambiente.py`](../src/verificar_ambiente.py).

Material relacionado:

- glossário do capítulo — [`technical-glossary/capitulo-01.md`](../technical-glossary/capitulo-01.md);
- mapa conceitual da arquitetura GPT — [`reports/sprint01-relatorio.md`](../reports/sprint01-relatorio.md).

---
## Capítulo 1 — Understanding Large Language Models

Leitura do livro, Capítulo 1. Buscar entender cada um destes pontos:

1. O que caracteriza um Large Language Model?
2. Qual a diferença entre IA, Machine Learning e Deep Learning?
3. Como os modelos GPT estão organizados?
4. Quais são as principais etapas do treinamento de um LLM?
5. Qual é o fluxo geral de funcionamento de um modelo de linguagem?

### 1. O que caracteriza um Large Language Model?

Um LLM é uma **rede neural profunda**, baseada na arquitetura **Transformer**, treinada em volumes muito
grandes de texto para processar, gerar e responder a textos em linguagem humana.

O "grande" (*large*) tem dois sentidos, e os dois caracterizam o modelo:

- **número de parâmetros** — os pesos ajustáveis da rede, que chegam a dezenas ou centenas de bilhões
  (o GPT-3 tem 175 bilhões);
- **tamanho do conjunto de treinamento** — corpora com centenas de bilhões de tokens (o GPT-3 foi
  treinado com 300 bilhões).

Além da escala, três traços distinguem um LLM dos modelos de NLP anteriores:

1. **Tarefa de treino simples e sem rótulos humanos**: prever a próxima palavra de um texto.
2. **Atenção**: o Transformer permite que o modelo dê pesos diferentes a cada parte da entrada ao
   fazer uma previsão, capturando contexto de longo alcance.
3. **Generalidade**: um único modelo executa muitas tarefas — tradução, resumo, classificação,
   redação —, enquanto os modelos anteriores eram construídos para uma tarefa cada.

Quando se diz que um LLM "entende" texto, o sentido é operacional: ele produz respostas coerentes e
adequadas ao contexto. Não se trata de consciência nem de compreensão no sentido humano.

### 2. Qual a diferença entre IA, Machine Learning e Deep Learning?

Não são três "tipos" independentes, e sim **campos aninhados**, um dentro do outro:

```
Inteligência Artificial
└── Machine Learning
    └── Deep Learning
        └── LLMs (aplicação de deep learning a texto)
```

- **Inteligência Artificial (IA)** — o campo mais amplo: construir sistemas que executam tarefas que
  exigiriam inteligência humana. Inclui abordagens que **não aprendem com dados**, como sistemas
  baseados em regras, sistemas especialistas, algoritmos genéticos, lógica fuzzy e raciocínio
  simbólico.
- **Machine Learning (ML)** — o ramo da IA em que o sistema **aprende a partir de dados**, em vez de
  ser programado explicitamente com as regras da tarefa. Exemplo: um filtro de spam que aprende com
  e-mails rotulados como spam ou legítimos, minimizando seus erros de previsão. No ML tradicional,
  porém, as **características** usadas pelo modelo (frequência de palavras como "grátis", número de
  pontos de exclamação, presença de links) são escolhidas por especialistas.
- **Deep Learning (DL)** — o ramo do ML que usa **redes neurais com três ou mais camadas** (redes
  profundas). A diferença prática é que a própria rede aprende quais características importam: **não
  há extração manual de características**. Os rótulos continuam necessários em tarefas
  supervisionadas; o que muda é quem decide o que observar nos dados.

Atenção ao termo "camadas": em deep learning ele se refere às camadas da rede neural, e não à relação
de hierarquia entre IA, ML e DL.

### 3. Como os modelos GPT estão organizados?

O GPT (*Generative Pretrained Transformer*) usa **apenas a parte de decoder** do Transformer original
— não há encoder. O Transformer de 2017 tinha um encoder, que lia a frase de entrada, e um decoder,
que gerava a tradução; o GPT descarta o encoder e fica só com a parte geradora.

A organização interna, da entrada para a saída:

1. **Tokenização e embeddings** — o texto vira uma sequência de tokens, cada token vira um vetor, e a
   esse vetor se soma uma informação de posição.
2. **Pilha de blocos Transformer** — o mesmo bloco se repete muitas vezes em sequência; cada bloco
   recebe a saída do anterior e a refina. Cada bloco combina **self-attention mascarada** (cada token
   só enxerga os anteriores) e uma **rede feed forward**, com normalização e conexões residuais
   (detalhes no Capítulo 4). O Transformer original repetia seus blocos 6 vezes; o GPT-3 tem 96.
3. **Camada de saída** — transforma o vetor de cada posição em uma pontuação para cada token do
   vocabulário, a partir da qual se escolhe o próximo token.

Duas consequências dessa organização:

- o GPT processa o texto em **um único sentido**, da esquerda para a direita;
- ele gera texto de forma **autorregressiva**: cada token produzido é anexado à entrada para produzir
  o próximo.

O contraste é o BERT, que usa só o encoder, lê a frase nos dois sentidos e é treinado para recuperar
palavras mascaradas — por isso é forte em classificação, mas não em geração. O mapa conceitual
completo está em [`reports/sprint01-relatorio.md`](../reports/sprint01-relatorio.md).

### 4. Quais são as principais etapas do treinamento de um LLM?

A sequência "aquisição de dados → pré-processamento → treinamento → ajuste fino" está correta, mas a
distinção central do capítulo é entre **duas etapas de treinamento com dados de naturezas
diferentes**:

1. **Coleta e preparação dos dados** — reunir um corpus grande e diverso de **texto bruto, não
   rotulado** (páginas web, livros, Wikipédia, código), filtrá-lo e tokenizá-lo.
2. **Pré-treinamento** — o modelo aprende a **prever a próxima palavra**. O rótulo de cada posição é a
   própria palavra seguinte do texto, então os rótulos são gerados automaticamente: é **aprendizado
   autossupervisionado**, e por isso é possível usar quantidades enormes de texto sem anotação humana.
   O resultado é um **modelo base** (*foundation model*), que sabe completar texto. É a etapa mais cara:
   o pré-treinamento do GPT-3 foi estimado em cerca de US$ 4,6 milhões em computação.
3. **Fine-tuning (ajuste fino)** — o modelo base continua sendo treinado em um conjunto **menor e
   rotulado**, específico de uma tarefa. As duas modalidades principais são:
   - **por instruções** — pares instrução → resposta (é assim que se obtém um assistente como o
     ChatGPT a partir do GPT-3);
   - **para classificação** — textos acompanhados de uma classe (por exemplo, spam / não spam).

No projeto, o pré-treinamento é feito em escala educacional na Sprint 5, e o fine-tuning na Sprint 6.

### 5. Qual é o fluxo geral de funcionamento de um modelo de linguagem?

A ideia original — o texto é dividido em tokens, o modelo processa esses tokens e gera um token de
saída que volta a ser texto — está correta. Detalhando cada passo:

1. **Texto → tokens**: o texto de entrada é quebrado em unidades (palavras, pedaços de palavras,
   pontuação).
2. **Tokens → token IDs**: cada token é trocado pelo seu número no vocabulário.
3. **Token IDs → embeddings**: cada ID vira um vetor, somado a um vetor que representa a posição do
   token.
4. **Blocos Transformer**: a self-attention mascarada permite a cada posição combinar informação dos
   tokens anteriores; as camadas seguintes refinam essa representação.
5. **Camada de saída → probabilidades**: o vetor da última posição é convertido em uma distribuição de
   probabilidade sobre todo o vocabulário.
6. **Escolha do próximo token**: seleciona-se um token (o mais provável, ou por sorteio) e ele é
   convertido de volta em texto.
7. **Laço autorregressivo**: o token gerado é **anexado à entrada**, e o processo recomeça do passo 3
   até se atingir o tamanho desejado ou um token de fim de texto.

O ponto que completa a resposta é o passo 7: o modelo gera **um token por vez**, e cada novo token é
escolhido condicionado a tudo o que veio antes.

---
## Apêndice A — Introdução ao PyTorch

Leitura do apêndice sobre PyTorch. Tópicos a compreender:

6. Criação de tensores;
7. Operações básicas;
8. Manipulação de dimensões;
9. Operações matriciais;
10. Funcionamento do autograd;
11. Construção de modelos utilizando módulos do PyTorch.

Os exemplos abaixo foram conferidos no ambiente do projeto; as saídas estão nos comentários. O script
[`src/verificar_ambiente.py`](../src/verificar_ambiente.py) executa uma versão de cada um deles.

### 6. Criação de tensores

Um **tensor** é a estrutura de dados central do PyTorch: um arranjo de números com qualquer número de
dimensões. Generaliza os conceitos da matemática — escalar (0 dimensões), vetor (1), matriz (2) e
tensores de ordem maior (3 ou mais). É parecido com um array do NumPy, mas pode rodar na GPU e
registrar operações para o cálculo de gradientes.

O **tipo de dado** (`dtype`) é inferido na criação: inteiros viram `int64` e números com casa decimal
viram `float32`. O `float32` é o padrão para redes neurais porque equilibra precisão e consumo de
memória, e as GPUs são otimizadas para ele.

```python
import torch

t0 = torch.tensor(1)                        # escalar — 0 dimensões
t1 = torch.tensor([1, 2, 3])                # vetor — 1 dimensão
t2 = torch.tensor([[1, 2], [3, 4]])         # matriz — 2 dimensões
t3 = torch.tensor([[[1, 2], [3, 4]],
                   [[5, 6], [7, 8]]])       # tensor de 3 dimensões, shape (2, 2, 2)

print(t1.dtype)                             # torch.int64
print(torch.tensor([1.0, 2.0]).dtype)       # torch.float32
print(t1.to(torch.float32).dtype)           # torch.float32 — conversão explícita
```

**No projeto:** os token IDs são tensores `int64`; os embeddings e os pesos do modelo são tensores
`float32` com shape `(lote, tokens, dimensão)`.

### 7. Operações básicas

As operações aritméticas (`+`, `-`, `*`, `/`) atuam **elemento a elemento**. Quando os shapes diferem,
o PyTorch aplica *broadcasting*: estende a dimensão menor para casar com a maior, sem copiar dados.
As **reduções** (`sum`, `mean`, `max`) resumem um tensor ao longo de uma dimensão escolhida em `dim`.
O atributo `.shape` informa as dimensões.

```python
a = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])

print(a.shape)           # torch.Size([2, 3]) — 2 linhas, 3 colunas
print(a + 10)            # broadcasting: tensor([[11., 12., 13.], [14., 15., 16.]])
print(a * a)             # elemento a elemento: tensor([[ 1., 4., 9.], [16., 25., 36.]])
print(a.sum(dim=1))      # soma de cada linha: tensor([ 6., 15.])
print(a.mean())          # média geral: tensor(3.5000)
```

**No projeto:** o softmax da atenção normaliza ao longo de `dim=-1` (cada linha da matriz de atenção
soma 1), e o embedding posicional é somado ao embedding de token por broadcasting.

### 8. Manipulação de dimensões

- `reshape(...)` e `view(...)` reorganizam os mesmos elementos em outro shape, **mantendo a ordem** em
  que estão na memória. `view` exige memória contígua; `reshape` copia os dados quando necessário.
- `.T` e `transpose(d0, d1)` **trocam eixos** — o que muda a ordem dos elementos.
- `unsqueeze(d)` acrescenta uma dimensão de tamanho 1; `squeeze(d)` remove.

A diferença entre reorganizar e transpor é fácil de confundir e aparece no resultado:

```python
a = torch.tensor([[1, 2, 3],
                  [4, 5, 6]])               # shape (2, 3)

print(a.reshape(3, 2))   # [[1, 2], [3, 4], [5, 6]] — mesma ordem, outro formato
print(a.view(3, 2))      # [[1, 2], [3, 4], [5, 6]] — idem
print(a.T)               # [[1, 4], [2, 5], [3, 6]] — linhas viram colunas

x = torch.rand(8, 4, 768)                  # (lote, tokens, dimensão)
print(x.transpose(1, 2).shape)             # torch.Size([8, 768, 4])
print(x.unsqueeze(0).shape)                # torch.Size([1, 8, 4, 768])
```

**No projeto:** a `MultiHeadAttention` da Sprint 3 usa `view` para dividir a dimensão em cabeças e
`transpose(1, 2)` para colocar as cabeças antes dos tokens.

### 9. Operações matriciais

A multiplicação de matrizes é feita com `matmul` ou com o operador `@`. A regra de shapes é
`(m × n) @ (n × p) → (m × p)`: a última dimensão do primeiro operando precisa ser igual à penúltima do
segundo. Com tensores de mais de 2 dimensões, as dimensões da frente são tratadas como **lote**, e a
multiplicação é aplicada às duas últimas — várias multiplicações em uma única chamada.

```python
a = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])            # (2, 3)

print(a @ a.T)                              # (2, 3) @ (3, 2) -> (2, 2)
# tensor([[14., 32.],
#         [32., 77.]])
print(torch.equal(a @ a.T, a.matmul(a.T)))  # True — mesma operação

q = torch.rand(2, 12, 4, 64)                # (lote, cabeças, tokens, head_dim)
k = torch.rand(2, 12, 4, 64)
print((q @ k.transpose(-2, -1)).shape)      # torch.Size([2, 12, 4, 4])
```

**No projeto:** é a operação mais usada do modelo. Os scores de atenção são `Q @ K.T`, e todas as
camadas lineares são multiplicações de matrizes.

### 10. Funcionamento do autograd

O **autograd** é o mecanismo de diferenciação automática do PyTorch. Quando um tensor é criado com
`requires_grad=True`, cada operação feita com ele é registrada em um **grafo de computação**. Ao
chamar `.backward()` sobre um resultado escalar (normalmente a perda), o PyTorch percorre esse grafo de
trás para frente, aplica a regra da cadeia e guarda em `.grad` a derivada da perda em relação a cada
tensor. É a retropropagação, sem precisar derivar nada à mão.

```python
x = torch.tensor([3.0], requires_grad=True)
y = (x ** 2).sum()            # y = x²
y.backward()
print(x.grad)                 # tensor([6.]) — dy/dx = 2x = 6

# Um "neurônio" linear com erro quadrático: perda = (w·entrada + b − alvo)²
w = torch.tensor([0.5], requires_grad=True)
b = torch.tensor([0.0], requires_grad=True)
entrada, alvo = torch.tensor([2.0]), torch.tensor([3.0])

perda = ((w * entrada + b - alvo) ** 2).mean()   # previsão 1,0; erro −2,0; perda 4,0
perda.backward()
print(w.grad, b.grad)         # tensor([-8.]) tensor([-4.])
# dperda/dw = 2·(previsão − alvo)·entrada = 2·(−2)·2 = −8
# dperda/db = 2·(previsão − alvo)          = 2·(−2)   = −4

with torch.no_grad():         # inferência: não registra o grafo, economiza memória
    z = w * entrada
print(z.requires_grad)        # False
```

**No projeto:** o laço de treinamento da Sprint 5 chama `perda.backward()` e depois deixa o otimizador
atualizar os pesos na direção oposta ao gradiente. Os experimentos da Sprint 3 já usaram o autograd
para medir o gradiente que atravessa o softmax.

### 11. Construção de modelos utilizando módulos do PyTorch

Modelos são construídos como subclasses de `torch.nn.Module`:

- no `__init__`, declaram-se as camadas — que registram automaticamente seus parâmetros;
- no `forward`, descreve-se como a entrada atravessa as camadas. O `backward` não precisa ser escrito:
  o autograd o deduz do `forward`.

`nn.Sequential` encadeia camadas que são aplicadas em ordem. `model.parameters()` dá acesso a todos os
pesos treináveis (`requires_grad=True`), e `model.train()` / `model.eval()` alternam entre os modos de
treino e de avaliação — o que importa para camadas como o dropout.

```python
import torch.nn as nn

class RedePequena(nn.Module):
    def __init__(self, n_entradas, n_saidas):
        super().__init__()
        self.camadas = nn.Sequential(
            nn.Linear(n_entradas, 4),   # 3·4 pesos + 4 vieses = 16 parâmetros
            nn.ReLU(),
            nn.Linear(4, n_saidas),     # 4·1 pesos + 1 viés  =  5 parâmetros
        )

    def forward(self, x):
        return self.camadas(x)

modelo = RedePequena(3, 1)
print(sum(p.numel() for p in modelo.parameters() if p.requires_grad))  # 21
print(modelo(torch.rand(2, 3)).shape)                                   # torch.Size([2, 1])
```

**No projeto:** todas as classes de [`src/attention.py`](../src/attention.py) seguem esse padrão, e o
modelo GPT da Sprint 4 será um `nn.Module` que reúne embeddings, blocos Transformer e camada de saída.